# Dual-Core VCO Tank Generation and EMX-DBS Setup

This notebook walks through the first intended `emx-dbs` structure family: a parameterized dual-core VCO tank seed. It generates a square-pixel N16-style GDS, previews the raw and configured layout, rasterizes it into optimizer masks, exports a candidate GDS, and finishes with a skeleton for a symmetry-aware DBS loop.

All generated files are written under `local/notebooks/dual_core_vco_tank/`, which is intentionally ignored by git.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import copy
import json
import sys

import numpy as np
import yaml
from IPython.display import Image, display

# Find the source checkout even when Jupyter starts in notebooks/.
repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / 'pyproject.toml').exists() and (candidate / 'emx_dbs').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Could not find the emx-dbs repository root')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

out_dir = repo_root / 'local' / 'notebooks' / 'dual_core_vco_tank'
out_dir.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {repo_root}')
print(f'Notebook outputs: {out_dir}')

In [ ]:
from emx_dbs.config import load_config, prepare_run_dir
from emx_dbs.corner_overlap import diagonal_bridge_centers
from emx_dbs.dbs import evaluate_masks
from emx_dbs.emx_runner import select_runner
from emx_dbs.gds_io import inspect_gds, inspect_raw_gds, write_candidate_gds
from emx_dbs.legality import check_legality
from emx_dbs.masks import apply_fixed_masks, save_masks_npz
from emx_dbs.mutate import apply_flips, sample_flip_group
from emx_dbs.rasterize import rasterize_config
from emx_dbs.reporting import write_gds_preview, write_layout_preview
from emx_dbs.tank_generator import (
    DualCoreVcoTankGeometry,
    generate_dual_core_vco_tank_gds,
    write_dual_core_vco_tank_config,
)


## 1. Generate A Square-Pixel Tank Seed

The generator emits the core layers we want for the first N16 tank studies by default: `M9` on `39/60`, `M8` on `38/40`, and `V8` on `58/60`. The feed labels are placed on the outer M9 feed edges as `PP`, `PN`, `SP`, and `SN`. This walkthrough also enables an optional fixed lower-metal guard ring on N16 M1 `31/0`; override the layer/datatype for another PDK if needed. The guard overlaps the north/south M9 feed edges and can carry optional `GS`/`GN` reference labels for EMX.

The code comments below describe the physical construction at a high level. The detailed polygon creation lives in `emx_dbs/tank_generator.py`.


In [ ]:
geom = DualCoreVcoTankGeometry(
    top_cell='dual_core_vco_tank_ring33',
    core_width_um=330.0,
    core_height_um=330.0,
    pitch_um=10.0,
    m9_ring_width_um=20.0,
    center_gap_um=10.0,
    feed_spacing_um=20.0,
    feed_width_um=5.0,
    feed_length_um=26.0,
    m8_trace_width_um=10.0,
    pixel_style='square',
    emit_v8=True,
    include_guard_ring=True,
    guard_layer=(31, 0),  # N16 M1; override for another PDK
    guard_offset_um=26.0,
    guard_width_um=10.0,
    guard_feed_overlap_um=5.0,  # overlap the north/south M9 feed edges in XY
    include_guard_ports=True,   # add static GS/GN guard reference labels
)

seed_gds = out_dir / 'ring33_square_guard_seed.gds'
config_path = out_dir / 'ring33_square_guard.local.yaml'

# Structure generation model:
# 1. Fill an M9 rectangular ring with pitch-sized square pixels.
# 2. Remove a central M9 gap from the top and bottom rails for differential access.
# 3. Add four narrow M9 feed rectangles and label PP/PN/SP/SN at the outer feed edges.
# 4. Add an M8 center trace on the adjacent lower metal.
# 5. Emit V8 wherever the seeded M8 and M9 masks overlap.
# 6. Add a fixed lower-metal guard ring whose north/south bars overlap the M9 feeds.
# 7. Optionally label the guard as GS/GN static reference ports.
#
# The generated YAML config is a local starting point for rasterization, export,
# and DBS. It defaults to the fake solver so this notebook can run off-server.
generate_dual_core_vco_tank_gds(seed_gds, geom)
write_dual_core_vco_tank_config(
    config_path,
    seed_gds,
    geom,
    run_id='notebook_ring33_square',
    output_root=out_dir / 'runs',
)

print(seed_gds)
print(config_path)


In [ ]:
raw_info = inspect_raw_gds(seed_gds, top_cell=geom.top_cell)
print(json.dumps(raw_info, indent=2, sort_keys=True))

raw_preview = out_dir / 'ring33_square_raw_preview.png'
write_gds_preview(seed_gds, raw_preview, top_cell=geom.top_cell, show_legend=True, show_title=False)
display(Image(filename=str(raw_preview)))


In [ ]:
cfg = load_config(config_path)
configured_info = inspect_gds(cfg)
print(json.dumps(configured_info, indent=2, sort_keys=True, default=str))

configured_preview = out_dir / 'ring33_square_configured_preview.png'
write_gds_preview(
    cfg.layout.seed_gds,
    configured_preview,
    top_cell=cfg.layout.top_cell,
    cfg=cfg,
    annotate_geometry=False,
    show_legend=True,
    show_title=False,
)
display(Image(filename=str(configured_preview)))


## 2. Rasterize, Visualize Masks, And Export A Candidate GDS

The optimizer works on boolean masks. Rasterization converts the configured GDS layers into one grid per logical layer, marks mutable pixels, preserves fixed regions, and records grid geometry for GDS export.

In [ ]:
maskset = rasterize_config(cfg)

seed_npz = out_dir / 'seed_masks.npz'
mask_preview = out_dir / 'seed_masks_preview.png'
exported_gds = out_dir / 'seed_exported_from_masks.gds'

save_masks_npz(maskset, seed_npz)
write_layout_preview(
    maskset,
    mask_preview,
    cfg,
    annotate_geometry=False,
    show_legend=True,
    show_title=False,
    show_fixed_overlay=False,
)
write_candidate_gds(maskset, cfg, exported_gds)

for layer_name, mask in maskset.masks.items():
    active = int(mask.sum())
    mutable = int(maskset.mutable_masks[layer_name].sum())
    print(f'{layer_name}: active={active}, mutable={mutable}, shape={mask.shape}')

print(f'Saved mask archive: {seed_npz}')
print(f'Exported GDS: {exported_gds}')
display(Image(filename=str(mask_preview)))


## 3. Sweep Geometry Parameters

Use `dataclasses.replace` to make controlled variants while keeping the layer mapping and base topology consistent. This is useful for seed studies before launching DBS.

In [ ]:
variants = {
    'narrow_ring': replace(geom, top_cell='tank_narrow_ring', m9_ring_width_um=10.0),
    'wide_ring': replace(geom, top_cell='tank_wide_ring', m9_ring_width_um=30.0),
    'octagon_reference': replace(
        geom,
        top_cell='tank_octagon_reference',
        pixel_style='octagon',
        include_octagon_corner_patches=True,
    ),
}

for name, variant_geom in variants.items():
    variant_gds = out_dir / f'{name}.gds'
    variant_png = out_dir / f'{name}.png'
    generate_dual_core_vco_tank_gds(variant_gds, variant_geom)
    write_gds_preview(variant_gds, variant_png, top_cell=variant_geom.top_cell)
    print(f'{name}: {variant_gds}')
    display(Image(filename=str(variant_png)))

## 4. Prepare A Real EMX Config Template

Keep process files, setup scripts, server paths, and process keys in ignored `*.local.yaml` files. The generated template below is deliberately not meant to be committed.

In [ ]:
real_cfg = yaml.safe_load(config_path.read_text())
real_cfg['run']['run_id'] = 'notebook_ring33_square_real_emx'
real_cfg['emx'].update(
    {
        'backend': 'real',
        'executable': 'emx',
        'proc_file': '/path/to/N16_or_site_process.integrand.proc',
        'env_script': '/path/to/local/setup_emx_env.sh',
        'key': '<process-file-key>',
        'freq_start_ghz': 1,
        'freq_stop_ghz': 20,
        'freq_step_ghz': 1,
        'timeout_s': 600,
        'retries': 2,
    }
)

real_config_path = out_dir / 'ring33_square.real_emx.template.local.yaml'
real_config_path.write_text(yaml.safe_dump(real_cfg, sort_keys=False), encoding='utf-8')
print(real_config_path)

## 5. Objective Definition Skeleton

`emx-dbs` objective plugins are ordinary Python functions with signature `fn(touchstone_path, metadata, params) -> ObjectiveResult`. Start with simple S-parameter terms, then replace or augment them with tank-specific quantities such as extracted differential inductance, Q, self-resonant frequency, balance, and area penalties.

In [ ]:
objective_module = out_dir / 'tank_objectives.py'
objective_source = """from pathlib import Path
import numpy as np
from emx_dbs.schemas import ObjectiveResult
from emx_dbs.touchstone import read_touchstone

def _mean(values, mask):
    selected = np.asarray(values)[mask]
    if selected.size == 0:
        return float('nan')
    return float(np.mean(selected))

def vco_tank_objective(touchstone_path, metadata, params):
    sp = read_touchstone(Path(touchstone_path))
    if sp.nports < 4:
        return ObjectiveResult(fom=-1e30, loss=1e30, valid=False, reason='requires_4_ports')
    mask = sp.band_mask(params.get('band_start_ghz'), params.get('band_stop_ghz'))
    # Port convention used by the generated seed: PP=1, PN=2, SP=3, SN=4.
    forward = 0.5 * (sp.mag(3, 1) + sp.mag(4, 2))
    reverse = 0.5 * (sp.mag(3, 2) + sp.mag(4, 1))
    return_mag = np.mean([sp.mag(p, p) for p in range(1, 5)], axis=0)
    mean_forward = _mean(np.clip(forward, 0.0, 1.0), mask)
    mean_reverse = _mean(np.clip(reverse, 0.0, 1.0), mask)
    mean_return = _mean(np.clip(return_mag, 0.0, 1.0), mask)
    target_forward = float(params.get('target_forward', 0.65))
    max_return = float(params.get('max_return', 0.25))
    reverse_weight = float(params.get('reverse_weight', 1.0))
    return_weight = float(params.get('return_weight', 1.0))
    loss = abs(target_forward - mean_forward)
    loss += reverse_weight * mean_reverse
    loss += return_weight * max(0.0, mean_return - max_return)
    valid = bool(np.isfinite(loss))
    return ObjectiveResult(
        fom=-loss,
        loss=loss,
        valid=valid,
        reason=None if valid else 'invalid_metric',
        metrics={
            'mean_forward': mean_forward,
            'mean_reverse': mean_reverse,
            'mean_return': mean_return,
        },
    )
"""
objective_module.write_text(objective_source, encoding='utf-8')

if str(out_dir) not in sys.path:
    sys.path.insert(0, str(out_dir))

objective_cfg = yaml.safe_load(config_path.read_text())
objective_cfg['objective'] = {
    'plugin': 'tank_objectives:vco_tank_objective',
    'params': {
        'band_start_ghz': 1,
        'band_stop_ghz': 20,
        'target_forward': 0.65,
        'max_return': 0.25,
        'reverse_weight': 1.0,
        'return_weight': 1.0,
    },
}

objective_config_path = out_dir / 'ring33_square.objective.local.yaml'
objective_config_path.write_text(yaml.safe_dump(objective_cfg, sort_keys=False), encoding='utf-8')
print(objective_module)
print(objective_config_path)

## 6. Symmetry Helpers For Pixelated DBS

For this tank family, the first DBS studies should mutate only `M9`, mirror every move about the y-axis, and keep the `M8` center trace plus `V8` overlap trace fixed. The helpers below do that on the current `MaskSet` representation.

Array rows correspond to y. Mirroring across the x-axis flips rows; mirroring across the y-axis flips columns.

In [ ]:
def symmetry_orbit(shape, row, col, mirror_x=False, mirror_y=True):
    """Return all array indices coupled by the selected mirror symmetries for one pixel."""
    nrows, ncols = shape
    points = {(int(row), int(col))}
    if mirror_x:
        points |= {(nrows - 1 - r, c) for r, c in list(points)}
    if mirror_y:
        points |= {(r, ncols - 1 - c) for r, c in list(points)}
    return sorted(points)


def mirrored_flips(maskset, flips, layers=None, mirror_x=False, mirror_y=True):
    """Expand sampled DBS flips into symmetry-linked flips."""
    selected_layers = set(layers) if layers is not None else set(maskset.masks)
    expanded = []
    seen = set()
    for layer, row, col in flips:
        if layer not in selected_layers:
            continue
        for rr, cc in symmetry_orbit(maskset.masks[layer].shape, row, col, mirror_x=mirror_x, mirror_y=mirror_y):
            key = (layer, rr, cc)
            if key not in seen and maskset.mutable_masks[layer][rr, cc]:
                expanded.append(key)
                seen.add(key)
    return expanded


def enforce_xy_symmetry(maskset, layers=('m9',), mirror_x=False, mirror_y=True):
    """Repair masks by OR-ing each selected layer with its x/y mirrors."""
    repaired = maskset.copy()
    for layer in layers:
        if layer not in repaired.masks:
            continue
        mask = repaired.masks[layer]
        symmetric = mask.copy()
        if mirror_x:
            symmetric |= mask[::-1, :]
        if mirror_y:
            symmetric |= mask[:, ::-1]
        if mirror_x and mirror_y:
            symmetric |= mask[::-1, ::-1]
        mutable = repaired.mutable_masks[layer]
        repaired.masks[layer][mutable] = symmetric[mutable]
    return apply_fixed_masks(repaired)


def sync_v8_to_metal_overlap(maskset, via_layer='v8', lower_layer='m8', upper_layer='m9'):
    """Regenerate mutable V8 pixels wherever lower and upper metals overlap."""
    required = {via_layer, lower_layer, upper_layer}
    if not required.issubset(maskset.masks):
        return maskset
    repaired = maskset.copy()
    via_grid = repaired.grids[via_layer]
    lower_grid = repaired.grids[lower_layer]
    upper_grid = repaired.grids[upper_layer]
    derived = np.zeros_like(repaired.masks[via_layer], dtype=bool)
    for row in range(derived.shape[0]):
        for col in range(derived.shape[1]):
            x_um, y_um = via_grid.index_center(row, col)
            lower_idx = lower_grid.xy_to_index(x_um, y_um)
            upper_idx = upper_grid.xy_to_index(x_um, y_um)
            if lower_idx is None or upper_idx is None:
                continue
            derived[row, col] = repaired.masks[lower_layer][lower_idx] and repaired.masks[upper_layer][upper_idx]
    mutable = repaired.mutable_masks[via_layer]
    repaired.masks[via_layer][mutable] = derived[mutable]
    return apply_fixed_masks(repaired)


# Demonstrate one symmetry-aware perturbation without invoking EMX.
rng = np.random.default_rng(geom.core_width_um.__int__())
sampled = sample_flip_group(maskset, cfg.dbs, rng)
m9_flips = [flip for flip in sampled if flip[0] == 'm9']
if not m9_flips:
    rows, cols = np.nonzero(maskset.mutable_masks['m9'])
    m9_flips = [('m9', int(rows[0]), int(cols[0]))]

candidate = apply_flips(maskset, mirrored_flips(maskset, m9_flips, layers={'m9'}, mirror_x=False, mirror_y=True))
candidate = enforce_xy_symmetry(candidate, layers=('m9',), mirror_x=False, mirror_y=True)
candidate = apply_fixed_masks(candidate)
assert np.array_equal(candidate.masks['m8'], maskset.masks['m8'])
assert np.array_equal(candidate.masks['v8'], maskset.masks['v8'])

candidate_preview = out_dir / 'symmetry_candidate_preview.png'
candidate_gds = out_dir / 'symmetry_candidate.gds'
write_layout_preview(
    candidate,
    candidate_preview,
    cfg,
    annotate_geometry=False,
    show_legend=True,
    show_title=False,
    show_fixed_overlay=False,
)
write_candidate_gds(candidate, cfg, candidate_gds)
print(f'Symmetric candidate GDS: {candidate_gds}')
display(Image(filename=str(candidate_preview)))

## 7. M9-Only DBS Trials With Corner Overlap

This section makes a few deterministic DBS-style M9-only candidate trials while copying M8 and V8 back from the seed after each move and enforcing y-axis symmetry. The full views keep only port labels/locations as annotations. The zoomed views show the exported corner-overlap bridge patches at the shared diagonal pixel corners.

Because V8 is fixed in these examples, M9 pixels directly under active V8 are excluded from the flip candidate list. The full legality check is also run before export, so candidates with `via_not_enclosed` are rejected instead of visualized as valid structures.


In [ ]:
trial_dir = out_dir / 'm9_only_dbs_trials'
trial_dir.mkdir(parents=True, exist_ok=True)


def m9_indices_under_active_v8(base):
    """Map fixed active V8 pixels to the M9 pixels that must remain on for enclosure."""
    locked = set()
    if not {'m9', 'v8'}.issubset(base.masks):
        return locked
    via_grid = base.grids['v8']
    m9_grid = base.grids['m9']
    rows, cols = np.nonzero(base.masks['v8'])
    for row, col in zip(rows.tolist(), cols.tolist()):
        x_um, y_um = via_grid.index_center(int(row), int(col))
        m9_idx = m9_grid.xy_to_index(x_um, y_um)
        if m9_idx is not None:
            locked.add((int(m9_idx[0]), int(m9_idx[1])))
    return locked


def m9_only_flip_candidates(base):
    """Return mutable M9 pixels away from feeds and fixed V8 enclosure pixels."""
    mutable = base.mutable_masks['m9'] & ~base.fixed_region_masks['m9']
    rows, cols = np.nonzero(mutable)
    nrows, ncols = base.masks['m9'].shape
    via_enclosure_locks = m9_indices_under_active_v8(base)
    candidates = []
    for row, col in zip(rows, cols):
        row, col = int(row), int(col)
        orbit = symmetry_orbit(mutable.shape, row, col, mirror_x=False, mirror_y=True)
        if not (4 <= row < nrows - 4 and 4 <= col < ncols - 4):
            continue
        if any((rr, cc) in via_enclosure_locks or not mutable[rr, cc] for rr, cc in orbit):
            continue
        candidates.append((row, col))
    return candidates


def y_mirrored_anchors(base, anchors):
    """Mirror 2x2 diagonal-contact anchors about the tank y-axis."""
    ncols = base.masks['m9'].shape[1]
    mirrored = set()
    flip_orientation = {'nw_se': 'ne_sw', 'ne_sw': 'nw_se'}
    for row, col, orientation in anchors:
        row, col = int(row), int(col)
        mirrored.add((row, col, orientation))
        mirrored.add((row, ncols - 2 - col, flip_orientation[orientation]))
    return sorted(mirrored)


def assert_y_axis_symmetric_mask(maskset, layer='m9'):
    if not np.array_equal(maskset.masks[layer], maskset.masks[layer][:, ::-1]):
        raise AssertionError(f'{layer} is not y-axis symmetric')


def impose_diagonal_contacts(candidate, anchors):
    """Force diagonal-only M9 contacts so the corner-overlap bridge exporter has visible work."""
    mask = candidate.masks['m9']
    mutable = candidate.mutable_masks['m9']
    for row, col, orientation in anchors:
        if orientation == 'nw_se':
            active = [(row, col), (row + 1, col + 1)]
            cleared = [(row, col + 1), (row + 1, col)]
        elif orientation == 'ne_sw':
            active = [(row, col + 1), (row + 1, col)]
            cleared = [(row, col), (row + 1, col + 1)]
        else:
            raise ValueError(f'Unknown diagonal orientation: {orientation}')
        for rr, cc in active + cleared:
            if not mutable[rr, cc]:
                raise RuntimeError(f'M9 anchor pixel {(rr, cc)} is not mutable')
        for rr, cc in active:
            mask[rr, cc] = True
        for rr, cc in cleared:
            mask[rr, cc] = False
    return candidate


def make_m9_only_trial(base, *, seed, anchors, random_flip_count=18):
    """Apply legal M9-only DBS-style flips, then restore M8/V8 exactly to seed masks."""
    rng = np.random.default_rng(seed)
    eligible = m9_only_flip_candidates(base)
    chosen = rng.choice(len(eligible), size=min(random_flip_count, len(eligible)), replace=False)
    raw_flips = [('m9', *eligible[int(idx)]) for idx in chosen]
    flips = mirrored_flips(base, raw_flips, layers={'m9'}, mirror_x=False, mirror_y=True)
    candidate = apply_flips(base, flips)
    candidate = impose_diagonal_contacts(candidate, y_mirrored_anchors(base, anchors))
    for fixed_layer in ('m8', 'v8'):
        candidate.masks[fixed_layer][:] = base.masks[fixed_layer]
        candidate.mutable_masks[fixed_layer][:] = False
    candidate = apply_fixed_masks(candidate)
    assert_y_axis_symmetric_mask(candidate, 'm9')
    reasons = check_legality(candidate, cfg).reasons
    if reasons:
        raise RuntimeError(f'Illegal M9-only trial candidate: {reasons}')
    return candidate, flips


trial_specs = [
    ('trial_00', 3301, [(10, 9, 'nw_se'), (14, 13, 'ne_sw')]),
    ('trial_01', 3302, [(11, 15, 'nw_se'), (16, 8, 'ne_sw'), (18, 18, 'nw_se')]),
    ('trial_02', 3303, [(12, 20, 'ne_sw'), (17, 12, 'nw_se'), (21, 15, 'ne_sw')]),
]

zoom_bounds_um = (70.0, 55.0, 235.0, 215.0)

for name, seed, anchors in trial_specs:
    trial, flips = make_m9_only_trial(maskset, seed=seed, anchors=anchors)
    assert np.array_equal(trial.masks['m8'], maskset.masks['m8'])
    assert np.array_equal(trial.masks['v8'], maskset.masks['v8'])

    overview_png = trial_dir / f'{name}_overview.png'
    zoom_png = trial_dir / f'{name}_corner_overlap_zoom.png'
    candidate_gds = trial_dir / f'{name}.gds'

    write_candidate_gds(trial, cfg, candidate_gds)
    write_layout_preview(
        trial,
        overview_png,
        cfg,
        annotate_geometry=False,
        show_legend=True,
        show_title=False,
        show_fixed_overlay=False,
    )
    write_layout_preview(
        trial,
        zoom_png,
        cfg,
        annotate_geometry=False,
        show_legend=True,
        show_title=False,
        show_fixed_overlay=False,
        bounds_um=zoom_bounds_um,
    )

    bridge_count = len(diagonal_bridge_centers(trial.masks['m9'], trial.grids['m9']))
    print(f'{name}: M9 flips={len(flips)}, M9 corner-overlap bridges={bridge_count}, GDS={candidate_gds}')
    display(Image(filename=str(overview_png)))
    display(Image(filename=str(zoom_png)))


## 8. Optimization Loop Skeleton

Use the normal CLI (`emx-dbs run ...`) for stock DBS. Use a custom loop like this when the move policy needs structure-specific repairs after every mutation. Leave `RUN_OPTIMIZATION = False` until the real EMX environment and objective are ready.


In [ ]:
RUN_OPTIMIZATION = False

if RUN_OPTIMIZATION:
    opt_cfg = load_config(objective_config_path)
    run_dir = prepare_run_dir(opt_cfg, objective_config_path)
    runner = select_runner(opt_cfg)
    rng = np.random.default_rng(opt_cfg.dbs.random_seed)

    current = rasterize_config(opt_cfg)
    current = enforce_xy_symmetry(current, layers=('m9',), mirror_x=False, mirror_y=True)
    current = apply_fixed_masks(current)

    best_fom = -1e30
    best_event = None
    rejections_in_a_row = 0

    for eval_index in range(opt_cfg.dbs.max_evaluations):
        if eval_index == 0:
            candidate = current.copy()
        else:
            flips = [flip for flip in sample_flip_group(current, opt_cfg.dbs, rng) if flip[0] == 'm9']
            flips = mirrored_flips(current, flips, layers={'m9'}, mirror_x=False, mirror_y=True)
            candidate = apply_flips(current, flips)
            candidate = enforce_xy_symmetry(candidate, layers=('m9',), mirror_x=False, mirror_y=True)
            candidate = apply_fixed_masks(candidate)

        event = evaluate_masks(
            opt_cfg,
            candidate,
            run_dir,
            eval_index,
            runner=runner,
            incumbent_fom=best_fom,
        )

        fom = float(event.get('fom', -1e30)) if event.get('objective_valid') else -1e30
        if fom > best_fom:
            current = candidate
            best_fom = fom
            best_event = event
            rejections_in_a_row = 0
        else:
            rejections_in_a_row += 1

        print(eval_index, 'fom=', fom, 'best=', best_fom, 'reason=', event.get('reason'))
        if rejections_in_a_row >= opt_cfg.dbs.max_rejections_in_a_row:
            break

    print('best_event:', best_event)
else:
    print('Optimization skeleton is disabled. Set RUN_OPTIMIZATION = True after EMX and the objective are ready.')

## 9. Robust DBS Configuration Starting Point

The settings below are a conservative starting point for a pixelated inverse-design loop: larger rejection budget, reproducible seed, corner-overlap bridges, explicit via stack, and optional forbidden shorts. A continuous M8 equator bar with V8 contacts on both sidewalls electrically connects the left and right M9 halves; if you enable primary/secondary feed-short constraints, use a no-short seed variant with the M8 center gap opened, as shown in the private N16 notebook. With `corner_overlap_bridge` enabled, diagonal-only same-layer pixel contacts are legal only because candidate GDS export writes a small diamond bridge at the shared pixel corner; the orange bridge markers in the preview correspond to that exported metal.


In [ ]:
robust_cfg = copy.deepcopy(objective_cfg)
robust_cfg['run']['run_id'] = 'notebook_ring33_square_symmetric_dbs'
robust_cfg['layout']['preserve_unconfigured_layers'] = False
robust_cfg['layout']['seed_vias_from_overlap'] = True
robust_cfg['connectivity'] = {
    # Use required groups only for nets that must remain connected.
    'required': [],
    # For a four-port differential tank, start by preventing obvious terminal shorts.
    # Tighten or relax this once the final port convention is fixed.
    'forbidden_shorts': [['PP', 'PN'], ['SP', 'SN'], ['PP', 'SN'], ['PN', 'SP']],
    'vias': [{'name': 'v8_stack', 'via_layer': 'v8', 'lower_layer': 'm8', 'upper_layer': 'm9'}],
}
robust_cfg['drc'] = {
    'min_width_um': 5.0,
    'min_spacing_um': 5.0,
    'allow_same_layer_diagonal_contact': True,
    'corner_overlap_bridge': True,
}
robust_cfg['dbs'] = {
    'max_evaluations': 250,
    'max_rejections_in_a_row': 80,
    'move_style': 'probabilistic_independent_layer_flips',
    'metal_flip_count_weights': [0.65, 0.25, 0.10],
    'metal_flip_count_values': [1, 2, 4],
    'random_seed': 33,
    'accept_equal': False,
}

robust_config_path = out_dir / 'ring33_square.symmetric_dbs.local.yaml'
robust_config_path.write_text(yaml.safe_dump(robust_cfg, sort_keys=False), encoding='utf-8')
print(robust_config_path)